# Bibliotecas

In [ ]:
# %pip install requests pandas

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 10.0/10.0 MB 66.9 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 12.6/12.6 MB 72.7 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [tzdata]
   ----- ---------------------------------- 1/8 [tzdata]
   ----- ---------------------------------- 1/8 [tzdata]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ----------------------------- 2/8 [numpy]
   ---------- ------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import requests
import pandas as pd
from IPython.display import HTML

In [17]:
BASE_URL = "https://pokeapi.co/api/v2"

def pegar_url_tipo(tipo):
    return f"https://raw.githubusercontent.com/duiker101/pokemon-type-svg-icons/master/icons/{tipo}.svg"

def pegar_icone_fraquezas(types):
    todos_tipos = [
        'normal', 'fire', 'water', 'grass', 'electric', 'ice', 
        'fighting', 'poison', 'ground', 'flying', 'psychic', 'bug', 
        'rock', 'ghost', 'dragon', 'steel', 'dark', 'fairy'
    ]
    multiplicador = {t: 1.0 for t in todos_tipos}

    for tipo_nome in types:
        type_res = requests.get(f"{BASE_URL}/type/{tipo_nome}")
        if type_res.status_code == 200:
            rel = type_res.json()['damage_relations']
            for t in rel['double_damage_from']:
                multiplicador[t['name']] *= 2.0
            for t in rel['half_damage_from']:
                multiplicador[t['name']] *= 0.5
            for t in rel['no_damage_from']:
                multiplicador[t['name']] *= 0.0

    icones = []
    fraquezas_ord = sorted(
        [(t, mult) for t, mult in multiplicador.items() if mult > 1.0],
        key=lambda x: x[1],
        reverse=True
    )

    for f_nome, mult in fraquezas_ord:
        tipo_url = pegar_url_tipo(f_nome)

        if mult == 4.0:
            icones_html = f'<span style="display: inline-block; border: 2px solid #ff0000; border-radius: 6px; padding: 2px 4px; margin: 2px;"><img src="{tipo_url}" height="18" width="18" style="vertical-align: middle;" title="Fraqueza 4x: {f_nome}"/> <b style="color: #cc0000; font-size: 11px; vertical-align: middle;">4x</b></span>'
        else:
            icones_html = f'<span style="display: inline-block; border: 1px solid #ccc; border-radius: 6px; padding: 2px; margin: 2px;"><img src="{tipo_url}" height="18" width="18" style="vertical-align: middle;" title="Fraqueza 2x: {f_nome}"/></span>'

        icones.append(icones_html)
    return "".join(icones) if icones else "Nenhuma"

def mini_html_sprite(nome_poke):
    res = requests.get(f"{BASE_URL}/pokemon/{nome_poke.lower()}")
    if res.status_code !=200:
        return f"<span>{nome_poke.capitalize()}<span>"

    data = res.json()
    sprite_url = data['sprites']['front_default']

    return f'<div style="display: inline-block; text-align: center; margin: 0 4px;"><img src="{sprite_url}" width="35" height="35" style="display: block; margin: 0 auto;"/><span style="font-size: 10px;">{nome_poke.capitalize()}</span></div>'

def info_evo_html(nome_poke):
    species_res = requests.get(f"{BASE_URL}/pokemon-species/{nome_poke}")
    if species_res.status_code !=200:
        return "-", "-"

    species_data = species_res.json()


    ante_evo = species_data.get('evolves_from_species')
    ante_html = mini_html_sprite(ante_evo['name']) if ante_evo else "-"


    chain_url = species_data['evolution_chain']['url']
    chain_res = requests.get(chain_url)
    if chain_res.status_code !=200:
        return ante_html, "-"

    chain_data = chain_res.json()['chain']
    prox_nomes = []

    def prox_evolucoes(node):
        if node['species']['name'] == nome_poke:
            for evo in node['evolves_to']:
                prox_nomes.append(evo['species']['name'])
        else:
            for evo in node['evolves_to']:
                prox_evolucoes(evo)
    prox_evolucoes(chain_data)

    if prox_nomes:
        prox_html = "".join([mini_html_sprite(nome) for nome in prox_nomes])
    else:
        prox_html = "-"

    return ante_html, prox_html

def prox_coluna_advanced(nome_ou_id):
    res = requests.get(f"{BASE_URL}/pokemon/{str(nome_ou_id).lower()}")
    if res.status_code !=200:
        return None

    data = res.json()
    nome_poke = data['name']

    sprite_url = data['sprites']['front_default']
    img_tag = f'<img src="{sprite_url}" width="60" />'

    #icone dos tipos
    tipos = [t['type']['name'] for t in data['types']]
    tipos_icones = []
    for t in tipos:
        icones_url = pegar_url_tipo(t)
        tipos_icones.append(f'<span style="display: inline-block; border: 1px solid #ddd; border-radius: 4px; padding: 2px; margin-right: 2px;"><img src="{icones_url}" height="20" width="20" title="{t}"/></span>')
    icones_tag = "".join(tipos_icones)

    # Icones de fraqueza
    tag_fraqueza = pegar_icone_fraquezas(tipos)

    # Evoluçoes
    ante_evo_html, prox_evo_html = info_evo_html(nome_poke)

    return {
        "ID": f"#{data['id']:03d}",
        "Imagem": img_tag,
        "Nome": nome_poke.capitalize(),
        "Tipos": icones_tag,
        "Fraquezas": tag_fraqueza,
        "Evolução Anterior": ante_evo_html,
        "Evolução Posterior": prox_evo_html
    }

In [19]:
entrada = input("Digite os Pokémons (nomes ou IDs) separados por vírgula: ")

# Transforma a resposta em uma lista de strings limpas (sem espaços extras)
poke = [p.strip() for p in entrada.split(",") if p.strip()]

dados = [prox_coluna_advanced(p) for p in poke if prox_coluna_advanced is not None]

df = pd.DataFrame(dados)

HTML(df.to_html(escape=False, index=False))

# lista_pokemons = ["charizard", "scizor", "eevee", "pupitar","yveltal", "gyarados", "terapagos"]

# dados = [prox_coluna_advanced(p) for p in lista_pokemons if prox_coluna_advanced(p) is not None]

# df = pd.DataFrame(dados)

# HTML(df.to_html(escape=False, index=False))

ID,Imagem,Nome,Tipos,Fraquezas,Evolução Anterior,Evolução Posterior
#445,,Garchomp,,4x,Gabite,-
